# Análisis Exploratorio de Datos (EDA)
**Tesis:** Modelo Predictivo de Resultados — Premier League 2017–2025  
**Dataset:** `model_dataset_clean.csv` generado por `pipeline_features.py`  
**Target:** `target_home_win` → 1 = Gana el local | 0 = Empate o gana el visitante

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('Dataset/model_dataset_clean.csv')
TARGET = 'target_home_win'
print(f'Shape: {df.shape}')
print(f'Temporadas: {sorted(df["season"].unique())}')
df.head(3)

---
## 1. Preprocesamiento y Variable Objetivo

In [ ]:
# El target ya está creado por pipeline_features.py
print('Distribución del target:')
print(df[TARGET].value_counts())
print(f'\n% victorias locales: {df[TARGET].mean()*100:.2f}%')
print(f'\nPartidos por temporada:')
print(df['season'].value_counts().sort_index())

---
## 2. Balance de Clases

In [ ]:
counts = df[TARGET].value_counts()
labels = ['No victoria local (0)', 'Victoria local (1)']
colores = ['#e07b71','#6baed6']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, counts.values, color=colores, edgecolor='white')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+15, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Balance de clases', fontsize=12)
axes[0].set_ylabel('Partidos')
axes[0].set_ylim(0, counts.max()*1.15)

axes[1].pie(counts.values, labels=labels, colors=colores, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción de clases', fontsize=12)

plt.tight_layout()
plt.savefig('plot_01_balance_clases.png', bbox_inches='tight')
plt.show()
print(f'Ratio desbalance: {counts[0]/counts[1]:.2f}:1')

---
## 3. Análisis Univariado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# odds_home_prob
sns.histplot(df['odds_home_prob'], bins=40, kde=True, color='#6baed6', ax=axes[0,0])
axes[0,0].axvline(df['odds_home_prob'].median(), color='red', linestyle='--',
                  label=f'Mediana: {df["odds_home_prob"].median():.2f}')
axes[0,0].set_title('Probabilidad implícita del local (odds_home_prob)')
axes[0,0].legend()

# diff_roll_points
sns.histplot(df['diff_roll_points'], bins=40, kde=True, color='#fd8d3c', ax=axes[0,1])
axes[0,1].axvline(0, color='black', linestyle='-', linewidth=1.2, label='Equilibrio de forma')
axes[0,1].axvline(df['diff_roll_points'].mean(), color='red', linestyle='--',
                  label=f'Media: {df["diff_roll_points"].mean():.2f}')
axes[0,1].set_title('Diferencia de forma reciente (diff_roll_points)')
axes[0,1].legend()

# home_unbeaten_streak
sns.histplot(df['home_unbeaten_streak'], bins=20, kde=False, color='#74c476', ax=axes[1,0])
axes[1,0].set_title('Racha invicta del equipo local')
axes[1,0].set_xlabel('Partidos consecutivos sin perder')

# away_unbeaten_streak
sns.histplot(df['away_unbeaten_streak'], bins=20, kde=False, color='#9e9ac8', ax=axes[1,1])
axes[1,1].set_title('Racha invicta del equipo visitante')
axes[1,1].set_xlabel('Partidos consecutivos sin perder')

plt.suptitle('Distribución de Features Clave', fontsize=13)
plt.tight_layout()
plt.savefig('plot_02_univariado.png', bbox_inches='tight')
plt.show()

---
## 4. Análisis de Multicolinealidad

In [ ]:
feat_cols = [c for c in df.columns if c not in ['season','date','home','away', TARGET]]
corr_m = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_m, dtype=bool))
sns.heatmap(corr_m, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1,
            linewidths=0.2, ax=ax, annot=False)
ax.set_title('Matriz de Correlación — Todas las Features', fontsize=13)
plt.tight_layout()
plt.savefig('plot_03_correlacion_matrix.png', bbox_inches='tight')
plt.show()

# Pares con alta colinealidad
UMBRAL = 0.80
ut = corr_m.where(np.triu(np.ones(corr_m.shape), k=1).astype(bool))
alta_col = (
    ut.stack().reset_index()
    .rename(columns={'level_0':'A','level_1':'B',0:'r'})
    .query('abs(r) > @UMBRAL')
    .sort_values('r', key=abs, ascending=False)
    .reset_index(drop=True)
)
print(f'Pares con |r| > {UMBRAL}: {len(alta_col)}')
print(alta_col.head(15).to_string())

---
## 5. Top Predictores (Correlación con Target)

In [ ]:
corr_target = (
    df[feat_cols + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .abs()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 7))
corr_target.head(20).sort_values().plot(kind='barh', color='#6baed6', edgecolor='white', ax=ax)
ax.set_title('Top 20 Predictores — Correlación Absoluta con target_home_win', fontsize=12)
ax.set_xlabel('|Correlación de Pearson|')
for bar in ax.patches:
    ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('plot_04_top_predictores.png', bbox_inches='tight')
plt.show()

print('Top 10 predictores:')
print(corr_target.head(10).round(4))

---
## 6. Baseline del Mercado de Apuestas

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

df_odds = df[df['odds_home_prob'] != df['odds_away_prob']].copy()
df_odds['market_pred'] = (df_odds['odds_home_prob'] > df_odds['odds_away_prob']).astype(int)
acc_market = accuracy_score(df_odds[TARGET], df_odds['market_pred'])

print(f'Partidos evaluados: {len(df_odds)}')
print(f'\n*** BASELINE DEL MERCADO: {acc_market:.4f} ({acc_market*100:.2f}%) ***')
print('\nReporte de clasificación:')
print(classification_report(df_odds[TARGET], df_odds['market_pred'],
      target_names=['No victoria local','Victoria local']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(df_odds[TARGET], df_odds['market_pred'])
ConfusionMatrixDisplay(cm, display_labels=['No victoria','Victoria local']).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Baseline del Mercado — Accuracy: {acc_market:.4f}', fontsize=10)
plt.tight_layout()
plt.savefig('plot_05_baseline_mercado.png', bbox_inches='tight')
plt.show()

---
## 7. Análisis Bivariado — Features vs Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
key_feats = ['odds_home_prob','diff_roll_points','away_unbeaten_streak',
             'home_roll_home_points','diff_season_gd','h2h_home_wins']

for ax, feat in zip(axes.flatten(), key_feats):
    for val, color, label in [(0,'#e07b71','No victoria'),(1,'#6baed6','Victoria local')]:
        sub = df[df[TARGET]==val][feat]
        ax.hist(sub, bins=25, alpha=0.55, color=color, label=label, density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)

plt.suptitle('Distribución de Features Clave por Resultado', fontsize=13)
plt.tight_layout()
plt.savefig('plot_06_bivariado.png', bbox_inches='tight')
plt.show()

---
## 8. Resumen del EDA

In [ ]:
print('='*60)
print('RESUMEN DEL EDA')
print('='*60)
print(f'Partidos analizados:    {len(df)}')
print(f'Temporadas:             2017-2018 a 2024-2025 (8 temporadas)')
print(f'Features disponibles:   {len(feat_cols)}')
print(f'Victorias locales:      {df[TARGET].sum()} ({df[TARGET].mean()*100:.1f}%)')
print(f'No victorias locales:   {(df[TARGET]==0).sum()} ({(df[TARGET]==0).mean()*100:.1f}%)')
print(f'Pares colineales (>0.8): {len(alta_col)}')
print(f'Top predictor:          {corr_target.index[0]} (|r|={corr_target.iloc[0]:.4f})')
print(f'Baseline mercado:       {acc_market*100:.2f}% accuracy')
print('='*60)